# Script para datos por horas

Este Script se encargar de extraer los datos que necesitamos, separados en hora y dia.

In [1]:
import pandas as pd
import numpy as np

## 1. Cargar todos los datos

In [2]:
files = [
    "../data/sample/2025-04-21_sample.csv",
    "../data/sample/2025-04-22_sample.csv",
    "../data/sample/2025-04-23_sample.csv",
    "../data/sample/2025-04-24_sample.csv",
    "../data/sample/2025-04-25_sample.csv",
    "../data/sample/2025-04-26_sample.csv",
    "../data/sample/2025-04-27_sample.csv"
]

In [4]:
viaje_dfs = []

for f in files:
    viaje_dfs.append(pd.read_csv(f))

viaje_dfs[1].head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada
0,BUS,0,2025-04-22 18:18:08,-,-,SAN RAMÓN,-,L-24-1-5-SN,-,-,-,347474,6287293,-,-
1,BUS,0,2025-04-22 08:28:24,-,-,MAIPÚ,-,L-13-74-5-PO,-,-,-,336937,6288490,-,-
2,METRO,1,2025-04-22 21:22:46,2025-04-22 21:33:20,634,LA REINA,LAS CONDES,PLAZA EGANA,FRANCISCO BILBAO,3030,2741,354016,6297357,352686,6299754
3,BUS,0,2025-04-22 21:35:38,-,-,PROVIDENCIA,-,E-14-116-OP-3,-,-,-,352617,6299684,-,-
4,BUS,1,2025-04-22 07:58:23,2025-04-22 08:08:17,594,RECOLETA,RECOLETA,L-4-4-45-PO,E-4-19-SN-55,1914,1084,346586,6303325,347126,6302401


## 2. Transformacion

transformar las columnas necesitamos en tipo correcto y agregar unos columnas que necesitamos.

In [5]:
for viaje_df in viaje_dfs:
    viaje_df.replace("-", np.nan, inplace=True)

    viaje_df["tiempo_subida"] = pd.to_datetime(viaje_df["tiempo_subida"])
    viaje_df['hora'] = viaje_df['tiempo_subida'].dt.hour

    viaje_df["tiene_bajada"] = viaje_df["tiene_bajada"].astype("bool")
    viaje_df["tiempo_etapa"] = viaje_df["tiempo_etapa"].astype("float")

## 3. Extraer

### 3.1 Cantidad de Pasajeros

In [6]:
pasajeros_dfs = []
for viaje_df, date in zip(viaje_dfs, range(21, 28)):

    pasajeros_por_hora = viaje_df.groupby('hora').size().reset_index(name=f'2025-04-{date}')

    # CODIGO SUGERIDO POR IA, PROMPT: "Como puedo asegurar que todas las 24 horas estén presentes"
    # Esto garantiza que el gráfico no tenga saltos si no hubo viajes a una hora específica.
    # Crea un índice de 0 a 23 y rellena las horas sin datos con 0.
    pasajeros_por_hora = pasajeros_por_hora.set_index('hora').reindex(range(24), fill_value=0).reset_index()
    # FIN CODIO IA

    pasajeros_dfs.append(pasajeros_por_hora)

In [7]:
df = pasajeros_dfs[0]
for i in pasajeros_dfs[1:]:
    df = df.merge(i, on="hora")
df.tail()

,hora,2025-04-21,2025-04-22,2025-04-23,2025-04-24,2025-04-25,2025-04-26,2025-04-27
19,19,304,326,320,335,272,339,409
20,20,200,203,186,206,202,256,369
21,21,101,139,144,134,145,198,229
22,22,77,66,65,87,93,120,113
23,23,9,11,14,17,19,23,10


In [8]:
df.to_csv(f"../data/pasajeros_por_hora.csv", index=False)

### 3.2 tiempo de Viaje

In [9]:
tiempo_dfs = []
for viaje_df, date in zip(viaje_dfs, range(21, 28)):
    has_bajada = viaje_df[viaje_df["tiene_bajada"] == True]

    tiempo_por_hora = has_bajada.groupby('hora').agg({"tiempo_etapa": "mean"}).reset_index()
    tiempo_por_hora.columns = ["hora", f'2025-04-{date}']

    # CODIGO SUGERIDO POR IA, PROMPT: "Como puedo asegurar que todas las 24 horas estén presentes"
    # Esto garantiza que el gráfico no tenga saltos si no hubo viajes a una hora específica.
    # Crea un índice de 0 a 23 y rellena las horas sin datos con 0.
    tiempo_por_hora = tiempo_por_hora.set_index('hora').reindex(range(24), fill_value=0).reset_index()
    # FIN CODIO IA

    tiempo_dfs.append(tiempo_por_hora)

In [10]:
df = tiempo_dfs[0]
for i in tiempo_dfs[1:]:
    df = df.merge(i, on="hora")
df.tail()

,hora,2025-04-21,2025-04-22,2025-04-23,2025-04-24,2025-04-25,2025-04-26,2025-04-27
19,19,1292.168182,1284.504132,1390.813559,1268.007937,1143.899441,1266.425743,1175.523077
20,20,1251.173333,1210.869863,1239.463768,1309.465753,1270.576389,1223.725146,1261.721519
21,21,998.844156,1085.972222,1006.637255,1083.505263,1246.160000,1129.844262,1127.913907
22,22,1281.638298,1189.854167,1333.562500,1297.603448,1343.163636,1200.382716,1245.838235
23,23,521.250000,1005.000000,895.875000,1026.400000,871.300000,743.888889,682.800000


In [11]:
df.to_csv(f"../data/tiempo_por_hora.csv", index=False)